# Catalog Exploration — Unity Catalog vs Legacy Hive Metastore
**Day 4 | Run on `dev-cluster`**

---

## What you see in the Catalog tab

```
Catalog tab
├── My organization
│     ├── dbw_ev_intelligence_dev   ← YOUR Unity Catalog — use this for everything
│     └── system                    ← Databricks internal (read-only, ignore)
├── Shares received
│     └── samples                   ← Delta Sharing demo data (read-only, ignore)
└── Legacy
      └── hive_metastore            ← OLD workspace Hive metastore (pre-UC)
            └── default             ← schema inside old metastore — DO NOT use
```

The **metastore** is the invisible account-level container above everything in "My organization".
It is not a clickable node — you see its catalogs, not the metastore object itself.

---

## What is `hive_metastore` (Legacy)?

When your workspace was first created, Databricks auto-created a workspace-level Hive metastore.
When Unity Catalog was enabled, Databricks moved it under **Legacy** instead of deleting it.

| | hive_metastore (Legacy) | dbw_ev_intelligence_dev (UC) |
|---|---|---|
| Namespace | `schema.table` (2-level) | `catalog.schema.table` (3-level) |
| Visible across workspaces | NO — this workspace only | YES — all workspaces in region |
| Column/row security | No | Yes |
| Data lineage | No | Yes |
| External Locations / Volumes | No | Yes |
| Permissions tab in UI | No | Yes |

**Rule: Never create tables under `hive_metastore` for this project.**

---

## Notebook structure

| Cell | What it does |
|---|---|
| 1 | Check which catalog Databricks defaults to (risk: may be hive_metastore) |
| 2 | Set catalog and schema to Unity Catalog — run this every notebook |
| 3 | List all catalogs with labels explaining each one |
| 4 | List schemas inside UC vs hive_metastore side by side |
| 5 | List tables and volumes inside your UC schema |
| 6 | Print scope comparison — hive_metastore vs Unity Catalog |
| 7 | Browse the Bronze Volume path |
| 8 | Full namespace reference card |


In [ ]:
# Cell 1 — Check the current default catalog
# Risk: if you run SQL without setting catalog, Databricks may default to hive_metastore.

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
current_schema  = spark.sql("SELECT current_schema()").collect()[0][0]

print(f"Current catalog : {current_catalog}")
print(f"Current schema  : {current_schema}")
print()

if current_catalog == "hive_metastore":
    print("WARNING: Default is hive_metastore (Legacy). Run Cell 2 to switch to Unity Catalog.")
elif current_catalog == "dbw_ev_intelligence_dev":
    print("OK: Already on Unity Catalog.")
else:
    print(f"INFO: On '{current_catalog}'. Run Cell 2 to set explicitly.")

In [ ]:
# Cell 2 — Set catalog and schema to Unity Catalog
# Run this at the top of EVERY notebook before any SQL or table operations.

spark.sql("USE CATALOG dbw_ev_intelligence_dev")
spark.sql("USE SCHEMA default")

print("Active catalog :", spark.sql("SELECT current_catalog()").collect()[0][0])
print("Active schema  :", spark.sql("SELECT current_schema()").collect()[0][0])
print()
print("All SQL now targets Unity Catalog. Short table names resolve to:")
print("  dbw_ev_intelligence_dev.default.<table_name>")

In [ ]:
# Cell 3 — List all catalogs with labels

print(f"  {'Catalog name':<35} Note")
print("  " + "-" * 70)
labels = {
    "hive_metastore":          "Legacy (pre-UC) — DO NOT use for new work",
    "system":                  "Databricks internal — read-only",
    "samples":                 "Delta Sharing demo — read-only",
    "dbw_ev_intelligence_dev": "YOUR Unity Catalog — use this for everything",
}
for row in spark.sql("SHOW CATALOGS").collect():
    name = row[0]
    print(f"  {name:<35} {labels.get(name, '')}")

In [ ]:
# Cell 4 — Schemas inside UC vs hive_metastore
# UC schemas are shared across workspaces. hive_metastore schemas exist in this workspace only.

print("Unity Catalog schemas (shared across all workspaces):")
for row in spark.sql("SHOW SCHEMAS IN dbw_ev_intelligence_dev").collect():
    print(f"  dbw_ev_intelligence_dev.{row[0]}")

print()
print("Legacy hive_metastore schemas (this workspace only — do not use):")
for row in spark.sql("SHOW SCHEMAS IN hive_metastore").collect():
    print(f"  hive_metastore.{row[0]}")

In [ ]:
# Cell 5 — Tables and volumes in your UC schema

print("TABLES in dbw_ev_intelligence_dev.default:")
tables = spark.sql("SHOW TABLES IN dbw_ev_intelligence_dev.default").collect()
if tables:
    for row in tables:
        print(f"  dbw_ev_intelligence_dev.default.{row['tableName']}")
else:
    print("  (none yet — will be created in Silver layer, Day 5+)")

print()
print("VOLUMES in dbw_ev_intelligence_dev.default:")
volumes = spark.sql("SHOW VOLUMES IN dbw_ev_intelligence_dev.default").collect()
if volumes:
    for row in volumes:
        print(f"  /Volumes/dbw_ev_intelligence_dev/default/{row['volume_name']}")
else:
    print("  (none yet)")

In [ ]:
# Cell 6 — Scope comparison: hive_metastore vs Unity Catalog

print("hive_metastore (Legacy):")
print("  Scope          : THIS workspace only")
print("  Namespace      : schema.table  (2-level)")
print("  Governance     : none — no column/row security, no lineage")
print("  Volumes / Ext  : not supported")
print("  Status         : deprecated")
print()
print("dbw_ev_intelligence_dev (Unity Catalog):")
print("  Scope          : ALL workspaces in the same metastore")
print("  Namespace      : catalog.schema.table  (3-level)")
print("  Governance     : column-level, row-level, lineage, permissions UI")
print("  Volumes / Ext  : yes — bronze-volume, evdatalakedev-bronze/silver/gold")
print("  Status         : current standard")
print()
print("VERDICT: Always use dbw_ev_intelligence_dev for this project.")

In [ ]:
# Cell 7 — Browse the Bronze Volume path
# /Volumes/... path works because UC External Location + Storage Credential handles auth.
# No ADLS credentials needed in this code.

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
print(f"Listing: {BRONZE_VOLUME}")
print()

try:
    for item in dbutils.fs.ls(BRONZE_VOLUME):
        print(f"  {'DIR ' if item.isDir() else 'FILE'}  {item.path}")
    print()
    print("Volume accessible — External Location + Storage Credential working correctly.")
except Exception as e:
    print(f"ERROR: {e}")
    print("Fix: GRANT READ VOLUME ON VOLUME dbw_ev_intelligence_dev.default.`bronze-volume`")
    print("     TO `your.email@company.com`;")

In [ ]:
# Cell 8 — Namespace reference card

print("UNITY CATALOG:")
print("  Catalog      : dbw_ev_intelligence_dev")
print("  Schema       : default")
print("  Table (SQL)  : dbw_ev_intelligence_dev.default.<table_name>")
print("  Volume (code): /Volumes/dbw_ev_intelligence_dev/default/bronze-volume/")
print()
print("ADLS PATHS (used by ADF, registered as UC External Locations):")
print("  Bronze : abfss://bronze@evdatalakedev.dfs.core.windows.net/")
print("  Silver : abfss://silver@evdatalakedev.dfs.core.windows.net/")
print("  Gold   : abfss://gold@evdatalakedev.dfs.core.windows.net/")
print()
print("SOURCE BLOB (read-only):")
print("  wasbs://source@dataenggdailystorage.blob.core.windows.net/")
print("  realtime/charging_sessions/YYYY/MM/DD/HH/*.csv")
print()
print("LEGACY (do not use):")
print("  hive_metastore.default.<table>  — workspace-only, no governance")